<a href="https://colab.research.google.com/github/Sonamgitdata/IIT-MAndi-Deep-learning/blob/main/assignment3_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part 3: Stress Testing & Robustness**

In [ ]:
import tensorflow as tf

# Load the MNIST dataset
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Load the MNIST dataset
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# ============================================================
# QUESTION 3.1
# VANISHING GRADIENTS & MODERN FIXES
# NumPy-only implementation
#
# Experiment A: Deep FCNN with Sigmoid
# Experiment B: Deep FCNN with ReLU + Batch Normalization
# ============================================================

np.random.seed(42)


# ============================================================
# 1. DATA PREPROCESSING
# ============================================================

# If MNIST images are in shape:
# (number_of_samples, 28, 28)
# convert them to:
# (number_of_samples, 784)

X_train = X_train.reshape(X_train.shape[0], -1).astype(np.float32)
X_test = X_test.reshape(X_test.shape[0], -1).astype(np.float32)

# Normalize pixel values from [0, 255] to [0, 1]
if X_train.max() > 1:
    X_train = X_train / 255.0
    X_test = X_test / 255.0


# ============================================================
# 2. ONE-HOT ENCODING
# ============================================================

def one_hot(y, num_classes=10):
    """
    Convert labels such as:
    [2, 5, 1]

    into:
    [
        [0,0,1,0,0,0,0,0,0,0],
        [0,0,0,0,0,1,0,0,0,0],
        [0,1,0,0,0,0,0,0,0,0]
    ]
    """
    y = y.astype(int)
    encoded = np.zeros((len(y), num_classes))
    encoded[np.arange(len(y)), y] = 1
    return encoded


Y_train = one_hot(y_train, 10)
Y_test = one_hot(y_test, 10)


# ============================================================
# 3. ACTIVATION FUNCTIONS
# ============================================================

def sigmoid(Z):
    """
    Sigmoid activation.
    Clip values to avoid numerical overflow.
    """
    Z = np.clip(Z, -500, 500)
    return 1.0 / (1.0 + np.exp(-Z))


def sigmoid_derivative(A):
    """
    Derivative of sigmoid.

    sigmoid'(z) = sigmoid(z) * (1 - sigmoid(z))

    Since A = sigmoid(Z):
    derivative = A * (1 - A)
    """
    return A * (1.0 - A)


def relu(Z):
    """
    ReLU activation:
    max(0, Z)
    """
    return np.maximum(0, Z)


def relu_derivative(Z):
    """
    Derivative of ReLU.
    """
    return (Z > 0).astype(float)


# ============================================================
# 4. SOFTMAX AND LOSS
# ============================================================

def softmax(Z):
    """
    Numerically stable Softmax.
    """
    Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)


def cross_entropy_loss(Y_true, Y_pred):
    """
    Cross-entropy loss.
    """
    epsilon = 1e-12

    loss = -np.sum(
        Y_true * np.log(Y_pred + epsilon)
    ) / Y_true.shape[0]

    return loss


# ============================================================
# 5. BATCH NORMALIZATION
# ============================================================

def batch_norm_forward(Z, gamma, beta, eps=1e-5):
    """
    Batch Normalization forward pass.

    Step 1: Calculate batch mean
    Step 2: Calculate batch variance
    Step 3: Normalize Z
    Step 4: Scale using gamma
    Step 5: Shift using beta
    """

    mean = np.mean(Z, axis=0, keepdims=True)
    variance = np.var(Z, axis=0, keepdims=True)

    # Normalize
    Z_normalized = (Z - mean) / np.sqrt(variance + eps)

    # Scale and shift
    BN_output = gamma * Z_normalized + beta

    cache = {
        "Z": Z,
        "Z_normalized": Z_normalized,
        "mean": mean,
        "variance": variance,
        "gamma": gamma,
        "beta": beta,
        "eps": eps
    }

    return BN_output, cache


def batch_norm_backward(dout, cache):
    """
    Batch Normalization backward pass.

    Returns:
    dZ     = gradient with respect to input Z
    dgamma = gradient with respect to gamma
    dbeta  = gradient with respect to beta
    """

    Z = cache["Z"]
    Z_normalized = cache["Z_normalized"]
    variance = cache["variance"]
    gamma = cache["gamma"]
    eps = cache["eps"]

    N = Z.shape[0]

    # Gradient for beta
    dbeta = np.sum(dout, axis=0, keepdims=True)

    # Gradient for gamma
    dgamma = np.sum(
        dout * Z_normalized,
        axis=0,
        keepdims=True
    )

    # Gradient with respect to normalized Z
    dZ_normalized = dout * gamma

    # Simplified BatchNorm backward formula
    std_inv = 1.0 / np.sqrt(variance + eps)

    dZ = (1.0 / N) * std_inv * (
        N * dZ_normalized
        - np.sum(dZ_normalized, axis=0, keepdims=True)
        - Z_normalized * np.sum(
            dZ_normalized * Z_normalized,
            axis=0,
            keepdims=True
        )
    )

    return dZ, dgamma, dbeta


# ============================================================
# 6. INITIALIZE NETWORK
# ============================================================

def initialize_network(layer_sizes, use_batchnorm=False):
    """
    Initialize all weights using He initialization.

    layer_sizes example:
    [784, 128, 128, 128, 128, 128, 128, 128, 128, 10]

    This means:
    Input = 784
    8 hidden layers = 128 neurons each
    Output = 10
    """

    parameters = {}

    number_of_layers = len(layer_sizes) - 1

    for layer in range(1, number_of_layers + 1):

        input_size = layer_sizes[layer - 1]
        output_size = layer_sizes[layer]

        # He initialization
        parameters[f"W{layer}"] = (
            np.random.randn(input_size, output_size)
            * np.sqrt(2.0 / input_size)
        )

        # Zero bias
        parameters[f"b{layer}"] = np.zeros(
            (1, output_size)
        )

        # BatchNorm is used only in hidden layers
        if use_batchnorm and layer < number_of_layers:

            # gamma starts from 1
            parameters[f"gamma{layer}"] = np.ones(
                (1, output_size)
            )

            # beta starts from 0
            parameters[f"beta{layer}"] = np.zeros(
                (1, output_size)
            )

    return parameters


# ============================================================
# 7. FORWARD PASS
# ============================================================

def forward_pass(
    X,
    parameters,
    activation="sigmoid",
    use_batchnorm=False
):
    """
    Forward propagation through the deep network.

    Experiment A:
    Dense -> Sigmoid

    Experiment B:
    Dense -> BatchNorm -> ReLU
    """

    caches = []

    number_of_layers = len(
        [key for key in parameters if key.startswith("W")]
    )

    A = X

    # --------------------------------------------------------
    # Hidden layers
    # --------------------------------------------------------

    for layer in range(1, number_of_layers):

        A_previous = A

        W = parameters[f"W{layer}"]
        b = parameters[f"b{layer}"]

        # Linear transformation
        Z = A_previous @ W + b

        bn_cache = None
        Z_before_activation = Z

        # Batch Normalization
        if use_batchnorm:

            gamma = parameters[f"gamma{layer}"]
            beta = parameters[f"beta{layer}"]

            Z, bn_cache = batch_norm_forward(
                Z,
                gamma,
                beta
            )

            Z_before_activation = Z

        # Activation
        if activation == "sigmoid":
            A = sigmoid(Z)

        elif activation == "relu":
            A = relu(Z)

        cache = {
            "A_previous": A_previous,
            "W": W,
            "b": b,
            "Z": Z_before_activation,
            "A": A,
            "bn_cache": bn_cache
        }

        caches.append(cache)

    # --------------------------------------------------------
    # Output layer
    # --------------------------------------------------------

    A_previous = A

    W = parameters[f"W{number_of_layers}"]
    b = parameters[f"b{number_of_layers}"]

    Z_output = A_previous @ W + b

    A_output = softmax(Z_output)

    output_cache = {
        "A_previous": A_previous,
        "W": W,
        "b": b,
        "Z": Z_output,
        "A": A_output
    }

    caches.append(output_cache)

    return A_output, caches


# ============================================================
# 8. BACKPROPAGATION
# ============================================================

def backward_pass(
    Y_true,
    Y_pred,
    caches,
    activation="sigmoid",
    use_batchnorm=False
):
    """
    Manual backpropagation.

    Calculates gradients for:
    W
    b
    gamma
    beta
    """

    gradients = {}

    number_of_layers = len(caches)
    batch_size = Y_true.shape[0]

    # --------------------------------------------------------
    # OUTPUT LAYER
    #
    # For Softmax + Cross Entropy:
    # dZ = Y_pred - Y_true
    # --------------------------------------------------------

    dZ = (Y_pred - Y_true) / batch_size

    output_cache = caches[-1]

    A_previous = output_cache["A_previous"]

    gradients[f"dW{number_of_layers}"] = (
        A_previous.T @ dZ
    )

    gradients[f"db{number_of_layers}"] = (
        np.sum(dZ, axis=0, keepdims=True)
    )

    # Gradient flowing to previous layer
    dA = dZ @ output_cache["W"].T

    # --------------------------------------------------------
    # HIDDEN LAYERS
    # --------------------------------------------------------

    for layer in reversed(range(1, number_of_layers)):

        cache = caches[layer - 1]

        A_previous = cache["A_previous"]
        W = cache["W"]
        Z = cache["Z"]
        A = cache["A"]

        # Activation backward
        if activation == "sigmoid":

            dZ_activation = dA * sigmoid_derivative(A)

        elif activation == "relu":

            dZ_activation = dA * relu_derivative(Z)

        # BatchNorm backward
        if use_batchnorm:

            dZ_linear, dgamma, dbeta = batch_norm_backward(
                dZ_activation,
                cache["bn_cache"]
            )

            gradients[f"dgamma{layer}"] = dgamma
            gradients[f"dbeta{layer}"] = dbeta

        else:
            dZ_linear = dZ_activation

        # Gradients for weights and bias
        gradients[f"dW{layer}"] = (
            A_previous.T @ dZ_linear
        )

        gradients[f"db{layer}"] = (
            np.sum(
                dZ_linear,
                axis=0,
                keepdims=True
            )
        )

        # Pass gradient to previous layer
        dA = dZ_linear @ W.T

    return gradients


# ============================================================
# 9. UPDATE PARAMETERS USING SGD
# ============================================================

def update_parameters(
    parameters,
    gradients,
    learning_rate,
    use_batchnorm=False
):
    """
    SGD update rule:

    W_new = W_old - learning_rate * dW
    """

    number_of_layers = len(
        [key for key in parameters if key.startswith("W")]
    )

    for layer in range(1, number_of_layers + 1):

        parameters[f"W{layer}"] -= (
            learning_rate
            * gradients[f"dW{layer}"]
        )

        parameters[f"b{layer}"] -= (
            learning_rate
            * gradients[f"db{layer}"]
        )

        # Update BatchNorm parameters
        if use_batchnorm and layer < number_of_layers:

            parameters[f"gamma{layer}"] -= (
                learning_rate
                * gradients[f"dgamma{layer}"]
            )

            parameters[f"beta{layer}"] -= (
                learning_rate
                * gradients[f"dbeta{layer}"]
            )

    return parameters


# ============================================================
# 10. ACCURACY FUNCTION
# ============================================================

def calculate_accuracy(Y_true, Y_pred):

    true_labels = np.argmax(Y_true, axis=1)
    predicted_labels = np.argmax(Y_pred, axis=1)

    accuracy = np.mean(
        true_labels == predicted_labels
    )

    return accuracy


# ============================================================
# 11. TRAINING FUNCTION
# ============================================================

def train_network(
    X_train,
    Y_train,
    X_test,
    Y_test,
    layer_sizes,
    activation="sigmoid",
    use_batchnorm=False,
    learning_rate=0.01,
    epochs=20,
    batch_size=128
):
    """
    Train the network and record:

    1. Training loss
    2. Training accuracy
    3. Test accuracy
    4. First-layer gradient norm
    """

    # Initialize model
    parameters = initialize_network(
        layer_sizes,
        use_batchnorm
    )

    # Store results
    training_losses = []
    training_accuracies = []
    test_accuracies = []
    first_layer_gradient_norms = []

    n_samples = X_train.shape[0]

    for epoch in range(epochs):

        # Shuffle training data
        indices = np.random.permutation(n_samples)

        X_shuffled = X_train[indices]
        Y_shuffled = Y_train[indices]

        epoch_gradient_norms = []

        # ----------------------------------------------------
        # MINI-BATCH TRAINING
        # ----------------------------------------------------

        for start in range(0, n_samples, batch_size):

            end = min(start + batch_size, n_samples)

            X_batch = X_shuffled[start:end]
            Y_batch = Y_shuffled[start:end]

            # Forward pass
            Y_pred, caches = forward_pass(
                X_batch,
                parameters,
                activation,
                use_batchnorm
            )

            # Backward pass
            gradients = backward_pass(
                Y_batch,
                Y_pred,
                caches,
                activation,
                use_batchnorm
            )

            # =================================================
            # IMPORTANT FOR QUESTION 3.1
            #
            # Calculate gradient norm of FIRST layer.
            #
            # ||dW1|| = sqrt(sum(dW1^2))
            # =================================================

            first_layer_grad_norm = np.linalg.norm(
                gradients["dW1"]
            )

            epoch_gradient_norms.append(
                first_layer_grad_norm
            )

            # Update parameters
            parameters = update_parameters(
                parameters,
                gradients,
                learning_rate,
                use_batchnorm
            )

        # ----------------------------------------------------
        # EVALUATE AFTER EACH EPOCH
        # ----------------------------------------------------

        # Training prediction
        train_predictions, _ = forward_pass(
            X_train,
            parameters,
            activation,
            use_batchnorm
        )

        train_loss = cross_entropy_loss(
            Y_train,
            train_predictions
        )

        train_accuracy = calculate_accuracy(
            Y_train,
            train_predictions
        )

        # Test prediction
        test_predictions, _ = forward_pass(
            X_test,
            parameters,
            activation,
            use_batchnorm
        )

        test_accuracy = calculate_accuracy(
            Y_test,
            test_predictions
        )

        # Save results
        training_losses.append(train_loss)
        training_accuracies.append(train_accuracy)
        test_accuracies.append(test_accuracy)

        # Average first-layer gradient norm for this epoch
        average_gradient_norm = np.mean(
            epoch_gradient_norms
        )

        first_layer_gradient_norms.append(
            average_gradient_norm
        )

        # Print results
        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy * 100:.2f}% | "
            f"Test Acc: {test_accuracy * 100:.2f}% | "
            f"First Layer Grad Norm: "
            f"{average_gradient_norm:.8f}"
        )

    return (
        parameters,
        training_losses,
        training_accuracies,
        test_accuracies,
        first_layer_gradient_norms
    )


# ============================================================
# 12. NETWORK ARCHITECTURE
# ============================================================

# Input = 784 pixels
# 8 hidden layers
# Output = 10 classes

layer_sizes = [
    784,

    # Hidden Layer 1
    128,

    # Hidden Layer 2
    128,

    # Hidden Layer 3
    128,

    # Hidden Layer 4
    128,

    # Hidden Layer 5
    128,

    # Hidden Layer 6
    128,

    # Hidden Layer 7
    128,

    # Hidden Layer 8
    128,

    # Output Layer
    10
]

print("Total hidden layers:", 8)


# ============================================================
# 13. EXPERIMENT A
# DEEP FCNN WITH SIGMOID
# ============================================================

print("\n")
print("=" * 60)
print("EXPERIMENT A: SIGMOID NETWORK")
print("=" * 60)

(
    sigmoid_parameters,
    sigmoid_losses,
    sigmoid_train_acc,
    sigmoid_test_acc,
    sigmoid_gradient_norms
) = train_network(
    X_train,
    Y_train,
    X_test,
    Y_test,
    layer_sizes=layer_sizes,
    activation="sigmoid",
    use_batchnorm=False,
    learning_rate=0.01,
    epochs=20,
    batch_size=128
)


# ============================================================
# 14. EXPERIMENT B
# DEEP FCNN WITH RELU + BATCH NORMALIZATION
# ============================================================

print("\n")
print("=" * 60)
print("EXPERIMENT B: RELU + BATCH NORMALIZATION")
print("=" * 60)

(
    relu_bn_parameters,
    relu_bn_losses,
    relu_bn_train_acc,
    relu_bn_test_acc,
    relu_bn_gradient_norms
) = train_network(
    X_train,
    Y_train,
    X_test,
    Y_test,
    layer_sizes=layer_sizes,
    activation="relu",
    use_batchnorm=True,
    learning_rate=0.01,
    epochs=20,
    batch_size=128
)


# ============================================================
# 15. PLOT 1:
# FIRST-LAYER GRADIENT NORM
# ============================================================

epochs = np.arange(1, len(sigmoid_gradient_norms) + 1)

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    sigmoid_gradient_norms,
    marker="o",
    label="Sigmoid"
)

plt.plot(
    epochs,
    relu_bn_gradient_norms,
    marker="o",
    label="ReLU + BatchNorm"
)

plt.xlabel("Epoch")
plt.ylabel("First Layer Gradient Norm")
plt.title(
    "Gradient Norm Comparison: First Layer"
)
plt.legend()
plt.grid(True)

plt.show()


# ============================================================
# 16. PLOT 2:
# TRAINING LOSS
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    sigmoid_losses,
    marker="o",
    label="Sigmoid"
)

plt.plot(
    epochs,
    relu_bn_losses,
    marker="o",
    label="ReLU + BatchNorm"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(
    "Training Loss Comparison"
)
plt.legend()
plt.grid(True)

plt.show()


# ============================================================
# 17. PLOT 3:
# TEST ACCURACY
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    np.array(sigmoid_test_acc) * 100,
    marker="o",
    label="Sigmoid"
)

plt.plot(
    epochs,
    np.array(relu_bn_test_acc) * 100,
    marker="o",
    label="ReLU + BatchNorm"
)

plt.xlabel("Epoch")
plt.ylabel("Test Accuracy (%)")
plt.title(
    "Test Accuracy Comparison"
)
plt.legend()
plt.grid(True)

plt.show()


# ============================================================
# 18. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

print("\nExperiment A: Sigmoid")
print(
    f"Final Training Accuracy: "
    f"{sigmoid_train_acc[-1] * 100:.2f}%"
)
print(
    f"Final Test Accuracy: "
    f"{sigmoid_test_acc[-1] * 100:.2f}%"
)
print(
    f"Final First-Layer Gradient Norm: "
    f"{sigmoid_gradient_norms[-1]:.8f}"
)

print("\nExperiment B: ReLU + BatchNorm")
print(
    f"Final Training Accuracy: "
    f"{relu_bn_train_acc[-1] * 100:.2f}%"
)
print(
    f"Final Test Accuracy: "
    f"{relu_bn_test_acc[-1] * 100:.2f}%"
)
print(
    f"Final First-Layer Gradient Norm: "
    f"{relu_bn_gradient_norms[-1]:.8f}"
)

Total hidden layers: 8


EXPERIMENT A: SIGMOID NETWORK
Epoch 01/20 | Loss: 2.3016 | Train Acc: 11.24% | Test Acc: 11.35% | First Layer Grad Norm: 0.00015645
Epoch 02/20 | Loss: 2.3014 | Train Acc: 11.24% | Test Acc: 11.35% | First Layer Grad Norm: 0.00015570
Epoch 03/20 | Loss: 2.3018 | Train Acc: 11.24% | Test Acc: 11.35% | First Layer Grad Norm: 0.00015565
Epoch 04/20 | Loss: 2.3018 | Train Acc: 11.24% | Test Acc: 11.35% | First Layer Grad Norm: 0.00015490
Epoch 05/20 | Loss: 2.3019 | Train Acc: 11.24% | Test Acc: 11.35% | First Layer Grad Norm: 0.00015533
Epoch 06/20 | Loss: 2.3016 | Train Acc: 11.24% | Test Acc: 11.35% | First Layer Grad Norm: 0.00015476
Epoch 07/20 | Loss: 2.3016 | Train Acc: 11.24% | Test Acc: 11.35% | First Layer Grad Norm: 0.00015565
Epoch 08/20 | Loss: 2.3018 | Train Acc: 10.44% | Test Acc: 10.28% | First Layer Grad Norm: 0.00015630
Epoch 09/20 | Loss: 2.3013 | Train Acc: 11.24% | Test Acc: 11.35% | First Layer Grad Norm: 0.00015548
Epoch 10/20 | Loss: 2.3020 